# 1D Burgers' Equation with FNO1D

Learn the operator that maps an initial velocity profile $u_0(x)$ to the solution $u(x, T)$ of the
viscous Burgers' equation with a 1D Fourier Neural Operator
([Li et al., 2021](https://arxiv.org/abs/2010.08895)):

$$u_t + u\, u_x = \nu\, u_{xx}, \quad x \in [0, 1) \text{ periodic}.$$

Burgers' equation is a standard nonlinear benchmark: its solutions steepen into shock-like fronts
that viscosity then smooths out.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`NU` is the viscosity and `T_END` the time the operator maps to.

In [ ]:
N_SAMPLES = 64
NX = 32
T_END = 0.5
NU = 0.02
MODES = 8
WIDTH = 32
DEPTH = 3
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01

## Load the data

`BurgersDataset` samples random initial conditions and integrates them to `T_END` with a
pseudospectral solver.

In [ ]:
import keras

from keras_sciml.datasets import BurgersDataset
from keras_sciml.losses import H1Loss, RelativeL2Loss
from keras_sciml.models import FNO1D

keras.utils.set_random_seed(42)

ds = BurgersDataset(mode="pseudospectral", n_samples=N_SAMPLES, nx=NX, t_end=T_END, nu=NU, seed=42)
u0, ut = ds.as_tuple()

n_train = int(0.8 * N_SAMPLES)
u0_train, ut_train = u0[:n_train], ut[:n_train]
u0_test, ut_test = u0[n_train:], ut[n_train:]
print(f"Train: {u0_train.shape}, Test: {u0_test.shape}")

## Define the model

The model is trained with an $H^1$ Sobolev loss, which penalises errors in the spatial derivative as
well as in the values. This helps the model get the steep fronts right.

In [ ]:
model = FNO1D(
    modes=MODES,
    width=WIDTH,
    depth=DEPTH,
    out_channels=1,
    activation="gelu",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=H1Loss(alpha=0.05, dx=1.0 / NX),
    metrics=[RelativeL2Loss(name="rel_l2")],
)

## Train

In [ ]:
history = model.fit(
    u0_train,
    ut_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(u0_test, ut_test),
    verbose=2,
)

## Evaluate

In [ ]:
test_loss = float(model.evaluate(u0_test, ut_test, verbose=0)[0])
print(f"Test H1 loss: {test_loss:.4f}")